In [0]:
!pip install kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 107.9 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os

os.environ["KAGGLE_USERNAME"] = "dnandi"
os.environ["KAGGLE_KEY"] = "KGAT_24e60868ea7a8fc8dc48dba5439a3692"

print("Kaggle credentials configured!")

Kaggle credentials configured!


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce
""")

DataFrame[]

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.ecommerce_data
""")

DataFrame[]

In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store

Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors


100%|██████████| 4.29G/4.29G [00:29<00:00, 158MB/s]


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
unzip -o ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

Archive:  ecommerce-behavior-data-from-multi-category-store.zip
  inflating: 2019-Nov.csv            
  inflating: 2019-Oct.csv            
total 18G
-rwxrwxrwx 1 spark-e12daa28-adb0-4083-974d-28 nogroup 8.4G Jan 19 08:09 2019-Nov.csv
-rwxrwxrwx 1 spark-e12daa28-adb0-4083-974d-28 nogroup 5.3G Jan 19 08:12 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 19 07:57 delta
-rwxrwxrwx 1 spark-e12daa28-adb0-4083-974d-28 nogroup 4.3G Jan 19 08:09 ecommerce-behavior-data-from-multi-category-store.zip
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 19 07:57 outputs


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
rm -f ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

total 14G
-rwxrwxrwx 1 spark-e12daa28-adb0-4083-974d-28 nogroup 8.4G Jan 19 08:09 2019-Nov.csv
-rwxrwxrwx 1 spark-e12daa28-adb0-4083-974d-28 nogroup 5.3G Jan 19 08:12 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 19 07:57 delta
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 19 07:57 outputs


In [0]:
%restart_python

In [0]:
from pyspark.sql import functions as F
csv_path = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv"
delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/delta/events_oct2019"

db = "workspace.ecommerce"
table_managed = f"{db}.events_oct2019_managed"
table_external = f"{db}.events_oct2019_external"

events = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(csv_path))

events = (events
          .withColumn("price", F.col("price").cast("double")))

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_gold")
base_vol = "/Volumes/workspace/ecommerce/ecommerce_data"

raw_csv = f"{base_vol}/2019-Oct.csv"

bronze_path = f"{base_vol}/delta/bronze/events"
silver_path = f"{base_vol}/delta/silver/events"
gold_path   = f"{base_vol}/delta/gold/product_perf"


In [0]:
bronze = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(raw_csv)
          .withColumn("ingestion_ts", F.current_timestamp())
          .withColumn("source_file", F.lit(raw_csv)))

(bronze.write
 .format("delta")
 .mode("overwrite")
 .save(bronze_path))

print("Bronze written to:", bronze_path)

Bronze written to: /Volumes/workspace/ecommerce/ecommerce_data/delta/bronze/events


In [0]:
(bronze.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("workspace.ecommerce_bronze.events"))

In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)

silver = (bronze_df
          .withColumn("event_ts", F.to_timestamp("event_time"))
          .withColumn("event_date", F.to_date("event_ts"))
          .withColumn("price", F.col("price").cast("double"))
          .filter(F.col("event_ts").isNotNull())
          .filter(F.col("event_type").isNotNull())
          .filter(F.col("user_session").isNotNull())
          .filter((F.col("price").isNull()) | ((F.col("price") > 0) & (F.col("price") < 10000)))
          .dropDuplicates(["user_session", "event_time", "event_type", "product_id"])
          .withColumn(
              "price_tier",
              F.when(F.col("price").isNull(), F.lit("unknown"))
               .when(F.col("price") < 10, F.lit("budget"))
               .when(F.col("price") < 50, F.lit("mid"))
               .otherwise(F.lit("premium"))
          )
         )

(silver.write
 .format("delta")
 .mode("overwrite")
 .save(silver_path))

print("Silver written to:", silver_path)

Silver written to: /Volumes/workspace/ecommerce/ecommerce_data/delta/silver/events


In [0]:
(silver.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("workspace.ecommerce_silver.events"))

In [0]:
silver_df = spark.read.format("delta").load(silver_path)

product_perf = (
    silver_df.groupBy("product_id")
    .agg(
        F.countDistinct(F.when(F.col("event_type") == "view", F.col("user_id"))).alias("unique_viewers"),
        F.countDistinct(F.when(F.col("event_type") == "purchase", F.col("user_id"))).alias("unique_purchasers"),
        F.round(F.sum(F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(F.lit(0.0))), 2).alias("revenue")
    )
    .withColumn(
        "conversion_rate_pct",
        F.when(F.col("unique_viewers") == 0, F.lit(0.0))
         .otherwise(F.round((F.col("unique_purchasers") / F.col("unique_viewers")) * 100, 4))
    )
)

(product_perf.write
 .format("delta")
 .mode("overwrite")
 .save(gold_path))

print("Gold written to:", gold_path)

display(product_perf.orderBy(F.col("revenue").desc()).limit(20))

Gold written to: /Volumes/workspace/ecommerce/ecommerce_data/delta/gold/product_perf


product_id,unique_viewers,unique_purchasers,revenue,conversion_rate_pct
1005115,170989,8352,1.240483595E7,4.8845
1005105,114813,4794,1.023924868E7,4.1755
1004249,96989,5538,6729380.83,5.7099
1005135,62646,2163,5567806.64,3.4527
1004767,175572,14410,5430222.72,8.2075
1002544,89025,6781,4854785.55,7.617
1004856,197840,19228,3798168.71,9.719
1002524,51704,4132,3538299.12,7.9916
1003317,56575,2179,3051294.26,3.8515
1004870,84318,7331,3027098.05,8.6945


In [0]:
(product_perf.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("workspace.ecommerce_gold.product_perf"))

In [0]:
dbutils.widgets.text("source_csv", "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv", "Source CSV")
dbutils.widgets.text("bronze_path", "/Volumes/workspace/ecommerce/ecommerce_data/delta/bronze/events", "Bronze path")
dbutils.widgets.text("bronze_table", "workspace.ecommerce_bronze.events", "Bronze table")

source_csv  = dbutils.widgets.get("source_csv")
bronze_path = dbutils.widgets.get("bronze_path")
bronze_table= dbutils.widgets.get("bronze_table")

In [0]:
raw = (spark.read.format("csv")
       .option("header","true")
       .option("inferSchema","true")
       .load(source_csv)
       .withColumn("ingestion_ts", F.current_timestamp())
       .withColumn("source_file", F.lit(source_csv)))

(raw.write.format("delta").mode("overwrite").save(bronze_path))
(raw.write.format("delta").mode("overwrite").saveAsTable(bronze_table))

print("Bronze done:", bronze_path, bronze_table)

Bronze done: /Volumes/workspace/ecommerce/ecommerce_data/delta/bronze/events workspace.ecommerce_bronze.events


In [0]:
dbutils.widgets.text("bronze_path", "/Volumes/workspace/ecommerce/ecommerce_data/delta/bronze/events", "Bronze path")
dbutils.widgets.text("silver_path", "/Volumes/workspace/ecommerce/ecommerce_data/delta/silver/events", "Silver path")
dbutils.widgets.text("silver_table", "workspace.ecommerce_silver.events", "Silver table")

bronze_path = dbutils.widgets.get("bronze_path")
silver_path = dbutils.widgets.get("silver_path")
silver_table= dbutils.widgets.get("silver_table")

In [0]:
bronze = spark.read.format("delta").load(bronze_path)

silver = (bronze
          .withColumn("event_ts", F.to_timestamp("event_time"))
          .withColumn("event_date", F.to_date("event_ts"))
          .withColumn("price", F.col("price").cast("double"))
          .filter(F.col("event_ts").isNotNull())
          .filter(F.col("user_session").isNotNull())
          .filter((F.col("price").isNull()) | ((F.col("price") > 0) & (F.col("price") < 10000)))
          .dropDuplicates(["user_session", "event_time", "event_type", "product_id"])
         )

(silver.write.format("delta").mode("overwrite").save(silver_path))
(silver.write.format("delta").mode("overwrite").saveAsTable(silver_table))

print("Silver done:", silver_path, silver_table)

Silver done: /Volumes/workspace/ecommerce/ecommerce_data/delta/silver/events workspace.ecommerce_silver.events


In [0]:
dbutils.widgets.text("silver_path", "/Volumes/workspace/ecommerce/ecommerce_data/delta/silver/events", "Silver path")
dbutils.widgets.text("gold_path", "/Volumes/workspace/ecommerce/ecommerce_data/delta/gold/product_perf", "Gold path")
dbutils.widgets.text("gold_table", "workspace.ecommerce_gold.product_perf", "Gold table")

silver_path = dbutils.widgets.get("silver_path")
gold_path   = dbutils.widgets.get("gold_path")
gold_table  = dbutils.widgets.get("gold_table")

In [0]:
silver = spark.read.format("delta").load(silver_path)

gold = (silver.groupBy("product_id")
        .agg(
            F.countDistinct(F.when(F.col("event_type") == "view", F.col("user_id"))).alias("unique_viewers"),
            F.countDistinct(F.when(F.col("event_type") == "purchase", F.col("user_id"))).alias("unique_purchasers"),
            F.round(F.sum(F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(F.lit(0.0))), 2).alias("revenue")
        )
        .withColumn(
            "conversion_rate_pct",
            F.when(F.col("unique_viewers") == 0, F.lit(0.0))
             .otherwise(F.round((F.col("unique_purchasers") / F.col("unique_viewers")) * 100, 4))
        )
       )

(gold.write.format("delta").mode("overwrite").save(gold_path))
(gold.write.format("delta").mode("overwrite").saveAsTable(gold_table))

print("Gold done:", gold_path, gold_table)

Gold done: /Volumes/workspace/ecommerce/ecommerce_data/delta/gold/product_perf workspace.ecommerce_gold.product_perf


In [0]:
silver_table = "workspace.ecommerce_silver.events"
silver_opt_table = "workspace.ecommerce_silver.events_part"   

In [0]:
print("Silver rows:", spark.table(silver_table).count())
display(spark.table(silver_table).groupBy("event_type").count().orderBy(F.col("count").desc()))

Silver rows: 42349871


event_type,count
view,40708806
cart,898292
purchase,742773


In [0]:
q_purchase = f"SELECT * FROM {silver_table} WHERE event_type = 'purchase'"
q_user     = f"SELECT * FROM {silver_table} WHERE user_id = 12345"

spark.sql(q_purchase).explain(True)
spark.sql(q_user).explain(True)

== Parsed Logical Plan ==
'Project [*]
+- 'Filter ('event_type = purchase)
   +- 'UnresolvedRelation [workspace, ecommerce_silver, events], [], false

== Analyzed Logical Plan ==
event_time: timestamp, event_type: string, product_id: int, category_id: bigint, category_code: string, brand: string, price: double, user_id: int, user_session: string, ingestion_ts: timestamp, source_file: string, event_ts: timestamp, event_date: date, price_tier: string
Project [event_time#17839, event_type#17840, product_id#17841, category_id#17842L, category_code#17843, brand#17844, price#17845, user_id#17846, user_session#17847, ingestion_ts#17848, source_file#17849, event_ts#17850, event_date#17851, price_tier#17852]
+- Filter (event_type#17840 = purchase)
   +- SubqueryAlias workspace.ecommerce_silver.events
      +- Relation workspace.ecommerce_silver.events[event_time#17839,event_type#17840,product_id#17841,category_id#17842L,category_code#17843,brand#17844,price#17845,user_id#17846,user_session#1784

In [0]:
import time

def bench_count(sql_text, runs=3):
    times = []
    for i in range(runs):
        t0 = time.time()
        spark.sql(sql_text).count()
        times.append(time.time() - t0)
    return times

baseline_purchase_times = bench_count(q_purchase, runs=3)
baseline_user_times     = bench_count(q_user, runs=3)

print("Baseline purchase query times (s):", baseline_purchase_times)
print("Baseline user_id query times (s):", baseline_user_times)

Baseline purchase query times (s): [0.9240484237670898, 0.6270771026611328, 0.736229419708252]
Baseline user_id query times (s): [0.5395951271057129, 0.4707942008972168, 0.49680376052856445]


In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {silver_opt_table}
USING DELTA
PARTITIONED BY (event_date, event_type)
AS
SELECT *
FROM {silver_table}
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql(f"""
OPTIMIZE {silver_opt_table}
ZORDER BY (user_id, product_id)
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
q_purchase_opt = f"SELECT * FROM {silver_opt_table} WHERE event_type = 'purchase'"
q_user_opt     = f"SELECT * FROM {silver_opt_table} WHERE user_id = 12345"

spark.sql(q_purchase_opt).explain(True)
spark.sql(q_user_opt).explain(True)

== Parsed Logical Plan ==
'Project [*]
+- 'Filter ('event_type = purchase)
   +- 'UnresolvedRelation [workspace, ecommerce_silver, events_part], [], false

== Analyzed Logical Plan ==
event_time: timestamp, event_type: string, product_id: int, category_id: bigint, category_code: string, brand: string, price: double, user_id: int, user_session: string, ingestion_ts: timestamp, source_file: string, event_ts: timestamp, event_date: date, price_tier: string
Project [event_time#19172, event_type#19173, product_id#19174, category_id#19175L, category_code#19176, brand#19177, price#19178, user_id#19179, user_session#19180, ingestion_ts#19181, source_file#19182, event_ts#19183, event_date#19184, price_tier#19185]
+- Filter (event_type#19173 = purchase)
   +- SubqueryAlias workspace.ecommerce_silver.events_part
      +- Relation workspace.ecommerce_silver.events_part[event_time#19172,event_type#19173,product_id#19174,category_id#19175L,category_code#19176,brand#19177,price#19178,user_id#19179,us

In [0]:
opt_purchase_times = bench_count(q_purchase_opt, runs=3)
opt_user_times     = bench_count(q_user_opt, runs=3)

print("Optimized purchase query times (s):", opt_purchase_times)
print("Optimized user_id query times (s):", opt_user_times)

print("\nAvg baseline purchase:", sum(baseline_purchase_times)/len(baseline_purchase_times))
print("Avg optimized purchase:", sum(opt_purchase_times)/len(opt_purchase_times))

print("\nAvg baseline user_id:", sum(baseline_user_times)/len(baseline_user_times))
print("Avg optimized user_id:", sum(opt_user_times)/len(opt_user_times))

Optimized purchase query times (s): [0.482593297958374, 0.41815733909606934, 0.4208076000213623]
Optimized user_id query times (s): [0.38825011253356934, 0.4766385555267334, 0.46082639694213867]

Avg baseline purchase: 0.7624516487121582
Avg optimized purchase: 0.44051941235860187

Avg baseline user_id: 0.5023976961771647
Avg optimized user_id: 0.44190502166748047
